# 05 — Frozen Error Cases, Grad-CAM and Final Analysis

**Research question:** Which spatial areas influence the separately fine-tuned ResNet18 models, and what limitations can reasonably be inferred from the observed errors?

Cases are selected **before heat maps are computed** using labels, probability, prediction, correctness, distance from threshold and stable sample IDs. For a binary one-logit model, the class-directed Grad-CAM target is **positive logit for `flip`** and **negative logit for `notflip`**. The README's 4×4 visualization displays correct `notflip`, correct `flip`, a high-confidence error and a near-threshold error in each regime **where these case types exist**. If a test set contains no case of a requested type, the panel says so instead of inventing a misclassification.

This notebook also selects up to 21 cases across regimes for the full interpretability analysis and calculates intermediate activation maps from four ResNet blocks. Attribution visualizations are descriptive, not proof of causal dependence.

In [ ]:
from pathlib import Path
import sys,pandas as pd
from IPython.display import display,Image as IPyImage
ROOT=Path.cwd().resolve();sys.path.insert(0,str(ROOT/'generated'))
import monreader_methods as mr
settings=mr.load_settings(ROOT);device=mr.setup(settings['seed'],settings['smoke'])

## Grad-CAM 4×4 preview

The following code loads each selected **frozen checkpoint** and the actual images corresponding to the test predictions, calculates layer-4 Grad-CAM, overlays it on the image, and saves the selection registry. Heat-map colours show positive normalized attribution magnitude to the nominated class; they do not establish that a physical feature caused the prediction. A smaller synthetic run can legitimately lack some of the error categories.

In [ ]:
camfile=mr.plot_cam_matrix(settings,device,Path(settings['out'])/'figures'/'resnet18_gradcam_4x4.png')
display(IPyImage(filename=str(camfile)))
reg=pd.read_csv(Path(settings['out'])/'figures'/'gradcam_16_panel_selection.csv')
display(reg[['regime','role','found','sample_id']])

## Full case registry and intermediate activations

The full registry targets 21 distinct case choices when the held-out populations contain the required examples. We retain the selection provenance and export one mean-absolute activation map per case at each of four ResNet stages. The synthetic smoke profile intentionally computes only a small number of these maps for speed; the full research profile exports the selected cases. When no specific role exists, the registry records the available cases rather than manufacturing absent examples.

In [ ]:
registry,activation=mr.generate_activation_diagnostics(settings,device,Path(settings['out'])/'figures'/'resnet18_intermediate_activations.png')
display(registry.head(10));display(activation.head(10))
display(IPyImage(filename=str(Path(settings['out'])/'figures'/'resnet18_intermediate_activations.png')))

## Technical conclusion and limits

The valid final interpretation uses only actual run outputs: independently fitted models, held-out predictions, source-bootstrap intervals, frame-level error records and class-directed visualizations. The four regimes are separate experimental designs, **not sequential fine-tuning**. A high held-out score does not guarantee robust new-source deployment. Cases and heatmaps illustrate hypotheses and failure patterns but cannot demonstrate their mechanisms causally. Research limitations include reconstruction of source IDs from filenames, correlated frames, recording coverage, optimizer randomness, and potentially scarce confident-error examples.

**Reproduction check:** notebooks 01–05 must execute in order from the same project directory with the correct dataset; the output folder contains split hashes, checkpoints, trial histories, per-frame predictions, figures and selection registries. No reported score is asserted to equal the author's published number. The notebook reader should compare their own measured outputs to any reported benchmark instead.

In [ ]:
summary=[]
for regime in mr.REGIMES:
    res=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
    scratch=pd.read_csv(Path(settings['out'])/'scratch'/regime/'heldout_test_predictions.csv')
    summary.append({'regime':regime,'scratch_test_macro_f1':mr.performance(scratch)['macro_f1'],
                    'resnet18_test_macro_f1':mr.performance(res)['macro_f1']})
summary=pd.DataFrame(summary)
display(summary)
summary.to_csv(Path(settings['out'])/'final_recomputed_summary.csv',index=False)
print('Complete: figures and results above were computed from this run, not pasted from historic tables.')